In [ ]:
import pickle
import os
import pandas as pd
from string import punctuation
import random

from nltk.tokenize import word_tokenize
from nltk.corpus import wordnet, stopwords
from nltk.stem import SnowballStemmer, WordNetLemmatizer
from nltk.tag import pos_tag
from nltk.probability import FreqDist
from nltk.classify import NaiveBayesClassifier, accuracy

### **Utility**

In [2]:
stemmer = SnowballStemmer('english')
lemmatizer = WordNetLemmatizer()
eng_stop = stopwords.words('english')

### **Preprocessing**

In [ ]:
def AlterTag (tag: str):
    if tag.startswith('J'):
        return 'a'
    elif tag.startswith('V'):
        return 'v'
    elif tag.startswith('R'):
        return 'r'
    return 'n'

def Preprocessing(docx: str):
    tokens = word_tokenize(docx.lower())
    tokens = [tok for tok in tokens if tok not in eng_stop]
    tokens = [tok for tok in tokens if tok not in punctuation]
    tokens = [tok for tok in tokens if tok.isalpha()]

    tagged = pos_tag(tokens)

    tokens = [lemmatizer.lemmatize(tok, AlterTag(tag)) for tok, tag in tagged]
    tokens = [stemmer.stem(tok) for tok in tokens]
    return tokens

### **Training**

In [ ]:
def Training():
    print('Initiate Training...')
    print('')
    data = pd.read_csv('./financial_dataset.csv')
    X = data['Statement']
    Y = data['Sentiment']

    # Feature
    feats = []

    for text, label in zip(X, Y):
        clean = Preprocessing(text)
        ft = dict(FreqDist(clean))

        # Atau Pakai ini:
        # Preferensi pribadi sih
        # ft = {word: True for word in clean}

        feats.append((ft, label))
    
    random.shuffle(feats)

    # Training
    print('Start Training...')
    split = int(len(feats) * 0.8)
    train_data = feats[:split]
    evals_data = feats[split:]

    model = NaiveBayesClassifier.train(train_data)
    acc = accuracy(model, evals_data)

    print('Model Trained')
    print(f'Accuracy: {acc}')
    print('')

    # Info
    print('Top 5 Most Informative Features')
    model.show_most_informative_features()
    print('')

    # Save
    with open('./model.pickle', 'wb') as file:
        pickle.dump(model, file)
    print('Model Saved')
    print('')

    return model


def Load():
    model = None
    if os.path.exists('./model.pickle'):
        with open('./model.pickle', 'rb') as file:
            model = pickle.load(file)
        return model
    else:
        print('No Model Detected')
        model = Training()
        return model

### **Support Function**

In [ ]:
def Write_State ():
    docx = ''

    while True:
        print('Enter your Statement: ')
        docx = input('>> ')

        if len(docx.split()) < 2:
            print('Please Enter at least 2 words')
        else:
            return docx
        
def Analyze_State (model, docx: str):
    if len(docx.split()) < 2:
        print('Please Enter your Statement First')
        return None
    
    